In [1]:
import os
import json
import openai

import data_utils

In [14]:
dataset = "cifar100"
prompt_type = "around"

In [15]:
openai.api_key = open(os.path.join(os.path.expanduser("~/Label-free-CBM"), ".openai_api_key"), "r").read()

In [16]:
prompts = {
    "important" : "List the most important features for recognizing something as a \"goldfish\":\n\n-bright orange color\n-a small, round body\n-a long, flowing tail\n-a small mouth\n-orange fins\n\nList the most important features for recognizing something as a \"beerglass\":\n\n-a tall, cylindrical shape\n-clear or translucent color\n-opening at the top\n-a sturdy base\n-a handle\n\nList the most important features for recognizing something as a \"{}\":",
    "superclass" : "Give superclasses for the word \"tench\":\n\n-fish\n-vertebrate\n-animal\n\nGive superclasses for the word \"beer glass\":\n\n-glass\n-container\n-object\n\nGive superclasses for the word \"{}\":",
    "around" : "List the things most commonly seen around a \"tench\":\n\n- a pond\n-fish\n-a net\n-a rod\n-a reel\n-a hook\n-bait\n\nList the things most commonly seen around a \"beer glass\":\n\n- beer\n-a bar\n-a coaster\n-a napkin\n-a straw\n-a lime\n-a person\n\nList the things most commonly seen around a \"{}\":"
}

base_prompt = prompts[prompt_type]

In [17]:
cls_file = data_utils.LABEL_FILES[dataset]
with open(cls_file, "r") as f:
    classes = f.read().split("\n")

In [18]:
feature_dict = {}

for i, label in enumerate(classes):
    feature_dict[label] = set()
    print("\n", i, label)
    for _ in range(2):
        response = openai.chat.completions.create(
              model="gpt-5-mini-2025-08-07",
              messages=[{"role": "user", "content": base_prompt.format(label)}],
              reasoning_effort= "minimal",
              verbosity="low",
              
              max_completion_tokens=256,
              
              frequency_penalty=0,
              presence_penalty=0
            )
        #clean up responses
        features = response.choices[0].message.content
        features = features.split("\n-")
        features = [feat.replace("\n", "") for feat in features]
        features = [feat.strip() for feat in features]
        features = [feat for feat in features if len(feat)>0]
        features = set(features)
        feature_dict[label].update(features)
    feature_dict[label] = sorted(list(feature_dict[label]))


 0 apple



 1 aquarium fish

 2 baby

 3 bear

 4 beaver

 5 bed

 6 bee

 7 beetle

 8 bicycle

 9 bottle

 10 bowl

 11 boy

 12 bridge

 13 bus

 14 butterfly

 15 camel

 16 can

 17 castle

 18 caterpillar

 19 cattle

 20 chair

 21 chimpanzee

 22 clock

 23 cloud

 24 cockroach

 25 couch

 26 crab

 27 crocodile

 28 cup

 29 dinosaur

 30 dolphin

 31 elephant

 32 flatfish

 33 forest

 34 fox

 35 girl

 36 hamster

 37 house

 38 kangaroo

 39 keyboard

 40 lamp

 41 lawn mower

 42 leopard

 43 lion

 44 lizard

 45 lobster

 46 man

 47 maple tree

 48 motorcycle

 49 mountain

 50 mouse

 51 mushroom

 52 oak tree

 53 orange

 54 orchid

 55 otter

 56 palm tree

 57 pear

 58 pickup truck

 59 pine tree

 60 plain

 61 plate

 62 poppy

 63 porcupine

 64 possum

 65 rabbit

 66 raccoon

 67 ray

 68 road

 69 rocket

 70 rose

 71 sea

 72 seal

 73 shark

 74 shrew

 75 skunk

 76 skyscraper

 77 snail

 78 snake

 79 spider

 80 squirrel

 81 streetcar

 82 sunflower

 83 sw

In [19]:
json_object = json.dumps(feature_dict, indent=4)
with open("data/concept_sets/gpt3_init/gpt3_{}_{}_new.json".format(dataset, prompt_type), "w") as outfile:
    outfile.write(json_object)